# Instance creation stats

This script generate a summary of how many Instance records were created by each staff user from a user-supplied number of days in the past until today.

## 1. Environment setup


In [ ]:
# !pip install pandas requests

import pandas as pd
import requests
from datetime import datetime, timedelta   

pd.set_option('display.max_columns', None)


## 2. Login
### Note: This script references patron data, so the login credentials must be able to access user accounts in FOLIO. 


In [ ]:

%run folio_auth.ipynb

## 3. A small helper for paginated GET requests

FOLIO endpoints typically page results via `limit`/`offset` query params and return a
JSON object with a records array plus a total count. This helper loops until it has
everything. **Confirm the pagination param names and the response envelope key names
against your instance** — I'm using common FOLIO conventions here, but I have not
verified them against current docs.


In [ ]:
def fetch_all_records(endpoint, records_key, limit=1000):
    """
    Fetch all records from a paginated FOLIO endpoint.

    endpoint: path like "/accounts", "/groups", "/users"
    records_key: the JSON key holding the list of records, e.g. "accounts", "usergroups", "users"
    """
    all_records = []
    offset = 0
    base_url = OKAPI_URL.copy()
    headers = HEADERS.copy()


    while True:
        response = requests.get(
            f"{base_url}{endpoint}",
            headers=headers,
            params={"limit": limit, "offset": offset},
        )
        response.raise_for_status()  # fail loudly and clearly if something's wrong
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break  # last page
        offset += limit

    return all_records

## 4. Get # of Days to Search from Input
The number entered will be used to search for Instance records with a date created of {number of days} - today's date. You can choose to bypass this and use a standard calculation if you prefer not to have an input.


In [ ]:

while True:
    try:
        from_days = int(input("Enter the number of days to search back from today, or press Enter to use the default of 90 days: ") or 90)
        if from_days < 0:  
            print("Number cannot be negative")
            continue
        else:                
            date_x_days_ago = datetime.now() - timedelta(days=from_days)
            search_date =str(date_x_days_ago.strftime("%Y-%m-%d"))
            print("You entered:", from_days, "days. The search will look for Instance records created since: ", search_date)
            break
    except ValueError:
        print("Please enter a valid number.")



## 4. Pull data from each endpoint

TODO (FOLIO-specific): confirm the `records_key` for each — FOLIO's convention is
usually the plural of the resource, but it varies (e.g. `/groups` often returns
`"usergroups"` rather than `"groups"` — **check this**, I'm not certain of the exact
key for your instance).


In [ ]:

def fetch_all_records(endpoint, records_key, limit=1000, query=None):
    all_records = []
    offset = 0
    base_url = OKAPI_URL
    headers = HEADERS.copy() if 'HEADERS' in globals() else {"X-Okapi-Tenant": TENANT, "Content-Type": "application/json"}
    if 'token' in globals():
        headers["Authorization"] = f"Bearer {token}"

    while True:
        params = {"limit": limit, "offset": offset}
        if query:
            params["query"] = query
        response = session.get(f"{base_url}{endpoint}", headers=headers, params=params)
        response.raise_for_status()
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break
        offset += limit

    return all_records


staff_raw = fetch_all_records(
    "/users",
    records_key="users",
    query='type=="staff"',
) 


instances_raw = fetch_all_records(
    "/instance-storage/instances",
    records_key="instances",
    query='metadata.createdDate >= ' + search_date,
)

print(f"staff:     {len(staff_raw)}")
print(f"instances: {len(instances_raw)}")


In [ ]:
# Inspect staff records
staff_df    = pd.DataFrame(staff_raw)
staff_df.head()

In [30]:
#Inspect instance records
instances_df   = pd.DataFrame(instances_raw)
instances_df.head()

,id,_version,hrid,source,title,indexTitle,alternativeTitles,editions,series,identifiers,contributors,subjects,classifications,publication,publicationFrequency,publicationRange,electronicAccess,dates,instanceTypeId,instanceFormatIds,instanceFormats,physicalDescriptions,languages,notes,administrativeNotes,modeOfIssuanceId,catalogedDate,previouslyHeld,staffSuppress,discoverySuppress,deleted,statisticalCodeIds,statusId,statusUpdatedDate,tags,metadata,holdingsRecords2,natureOfContentTermIds
0,fff6cbc4-89d8-4b11-912e-899988eef596,1,in00000425095,MARC,12 things to know about political parties / Vi...,12 things to know about political parties /,[{'alternativeTitleTypeId': 'd84bde11-0b9e-4ae...,[],[],"[{'value': '23766617', 'identifierTypeId': '7e...","[{'name': 'Hayes, Vicki C.', 'contributorTypeI...",[{'value': 'Political parties--United States--...,"[{'classificationNumber': 'JK2261 .H337 2025',...","[{'publisher': 'Black Rabbit Books', 'place': ...",[],[],[],{'dateTypeId': '24a506e8-2a92-4ecc-bd09-ff8493...,6312d172-f0cf-40f6-b27d-9fa8feaf332f,[8d511d33-5e85-4c5d-9bce-6e3c9cd0c324],[],[48 pages : illustrations ; 24 cm.],[eng],[{'instanceNoteTypeId': '86b6e817-e1bc-42fb-ba...,[],9d18a02f-5897-4c31-9106-c9abb5c7ae8b,2026-07-20,False,False,False,False,[],dd52773c-cfc0-4390-9551-6f98444d7bfa,2026-07-20T18:12:50.827+0000,{'tagList': []},{'createdDate': '2026-07-20T18:12:50.826+00:00...,[],[]
1,4852f533-06d9-413e-b7c3-f97586631a4f,1,in00000425128,MARC,Writing the U.S. Constitution : framework of a...,Writing the U.S. Constitution : framework of a...,[{'alternativeTitleTypeId': 'd84bde11-0b9e-4ae...,[],[{'value': 'Perspectives library.'}],"[{'value': 'in00024787139', 'identifierTypeId'...","[{'name': 'Baxter, Roberta, 1952-', 'contribut...",[{'value': 'United States. Constitution--Juven...,"[{'classificationNumber': '342.73029', 'classi...","[{'publisher': 'Cherry Lake Press', 'place': '...",[],[],[],{'dateTypeId': '24a506e8-2a92-4ecc-bd09-ff8493...,6312d172-f0cf-40f6-b27d-9fa8feaf332f,[8d511d33-5e85-4c5d-9bce-6e3c9cd0c324],[],[32 pages: illustrations (some color) ; 24 cm.],[eng],[{'instanceNoteTypeId': '6a2533a7-4de2-4e64-84...,[],9d18a02f-5897-4c31-9106-c9abb5c7ae8b,2026-07-21,False,False,False,False,[],dd52773c-cfc0-4390-9551-6f98444d7bfa,2026-07-21T13:51:44.819+0000,{'tagList': []},{'createdDate': '2026-07-21T13:51:44.818+00:00...,[],[]
2,d7b6c2a9-95ed-4856-91a2-647c8298e97a,1,in00000425230,MARC,Bicycling the Colorado Rockies / Vici De Haan.,Bicycling the Colorado Rockies /,[],"[Rev. & expanded, 2nd rd.]",[],"[{'value': '4749869', 'identifierTypeId': '7e5...","[{'name': 'DeHaan, Vici', 'contributorNameType...",[{'value': 'Bicycle touring--Rocky Mountains--...,[{'classificationNumber': 'GV1045.5.R6 D4 1985...,"[{'publisher': 'Pruett Pub. Co.', 'place': 'Bo...",[],[],[],{'dateTypeId': '24a506e8-2a92-4ecc-bd09-ff8493...,6312d172-f0cf-40f6-b27d-9fa8feaf332f,[8d511d33-5e85-4c5d-9bce-6e3c9cd0c324],[],[114 p. : ill. ; 22 cm.],[eng],[{'instanceNoteTypeId': '86b6e817-e1bc-42fb-ba...,[],9d18a02f-5897-4c31-9106-c9abb5c7ae8b,NaN,False,False,False,False,[],NaN,2026-07-22T20:15:00.786+0000,{'tagList': []},{'createdDate': '2026-07-22T20:15:00.785+00:00...,[],[]
3,2f756088-c4ad-41ff-8c0b-3936d9d9d8ed,2,in00000425296,MARC,Tōkyō Kokuritsu Hakubutsukan zuhan mokuroku....,Tōkyō Kokuritsu Hakubutsukan zuhan mokuroku....,[{'alternativeTitleTypeId': 'd84bde11-0b9e-4ae...,[],[],"[{'value': '5036355', 'identifierTypeId': '7e5...","[{'name': 'Tōkyō Kokuritsu Hakubutsukan', 'c...",[{'value': 'Tōkyō Kokuritsu Hakubutsukan--Ca...,"[{'classificationNumber': 'ND1055 .T627 1984',...",[{'publisher': 'Tōkyō Kokuritsu Hakubutsukan...,[],[],[],{'dateTypeId': '24a506e8-2a92-4ecc-bd09-ff8493...,6312d172-f0cf-40f6-b27d-9fa8feaf332f,[8d511d33-5e85-4c5d-9bce-6e3c9cd0c324],[],"[x, 116 p. : chiefly ill. ; 27 cm.]","[jpn, Jpn, eng]",[],[],9d18a02f-5897-4c31-9106-c9abb5c7ae8b,NaN,False,False,False,False,[],NaN,2026-07-24T23:51:56.138+0000,{'tagList': []},{'createdDate'

## 5. Inspect join keys before merging

This is the step worth slowing down for. Before merging, check:
- Do the key columns actually exist under the names you expect?
- Are the data types consistent between the two sides of each join (e.g. both strings)?
- Any leading/trailing whitespace or case differences?


In [ ]:
print(staff_df.columns.tolist())
print(instances_df.columns.tolist())

In [ ]:
# Spot-check types of the columns you intend to join on
# Since the user ID in the instance records is nested in the metadata dictionary, we need to extract it first
print(staff_df['id'].dtype)
print(instances_df['metadata'].apply(lambda x: x.get('createdByUserId') if isinstance(x, dict) else None).dtype)

## 6. Merge

Two joins: accounts → users, then that result → groups.

Starting with `how='left'` keeps every account row even if a match isn't found, so you
can see what didn't match rather than silently losing rows.


In [ ]:
instances_users = instances_df.merge(staff_df, left_on=instances_df['metadata'].apply(lambda x: x.get('createdByUserId') if isinstance(x, dict) else None), right_on='id', how='left', suffixes=('_instance', '_user')
)
instances_users.head()

## 7. Validate the merge

Common beginner pitfall: a one-to-many relationship silently multiplying rows, or a
type mismatch causing everything to come back unmatched. Check both.


In [ ]:
print("Original Instance rows:", len(instances_df))
print("After merging with staff:  ", len(instances_users))

## 8. Analyze the combined dataset

Now that accounts, users, and groups are joined, you can ask questions that span all
three — e.g. total fee/fine amounts by patron group. Adjust field names to match your
actual `/accounts` schema (commonly something like `amount` or `remaining`).


In [ ]:
summary=instances_users.groupby('username')['id_instance'].count().sort_values(ascending=False)
print("Summary of instance records created by each staff user from", search_date, "to today:")
print(summary)

In [28]:
# expanded_staff=instances_users['personal'].apply(pd.Series)
# expanded_staff=pd.json_normalize(instances_users['personal'])
# print("Expanded staff personal information:")
# expanded_staff.head()

flattened_st = pd.DataFrame()
flattened_st['username'] = staff_df['username']
flattened_st['firstName'] = staff_df['personal'].apply(lambda x: x.get('firstName') if isinstance(x, dict) else None)
flattened_st['lastName'] = staff_df['personal'].apply(lambda x: x.get('lastName') if isinstance(x, dict) else None)
flattened_st['staffId']= staff_df['id']
print("Basic staff information:")
flattened_st.head()


Basic staff information:


,username,firstName,lastName,staffId
0,scurry,Stephen,Curry,a8046409-430a-41f0-87a0-9c7e0a19f9ee
1,postman,Postman,User,1bc06b9d-0e6e-4ac2-91e1-1fb6677ba6f5
2,EBSCOGCSarainio,EBSCOGCSarainio,EBSCOGCSarainio,c59023c4-7838-4e67-a028-9c143123dda4
3,chutchinson,Corrie,Hutchinson,5e20afa6-13e3-484b-b861-9956204c9ba4
4,ibrown,Isabella,Brown,c1c0f905-e01f-49d3-9eb4-d8ae44b9416c


In [31]:
flat_inst = pd.DataFrame()
flat_inst['instanceId'] = instances_df['id']
flat_inst['hrid'] = instances_df['hrid']
flat_inst['title'] = instances_df['title']
flat_inst['createdByUserId'] = instances_df['metadata'].apply(lambda x: x.get('createdByUserId') if isinstance(x,dict) else None)
print("Basic Instance information")
print(flat_inst)

Basic Instance information
                               instanceId           hrid  \
0    fff6cbc4-89d8-4b11-912e-899988eef596  in00000425095   
1    4852f533-06d9-413e-b7c3-f97586631a4f  in00000425128   
2    d7b6c2a9-95ed-4856-91a2-647c8298e97a  in00000425230   
3    2f756088-c4ad-41ff-8c0b-3936d9d9d8ed  in00000425296   
4    0095be4b-cc88-4cd5-979c-62b8b7da49ce  in00000425161   
..                                    ...            ...   
105  195c6d1a-3384-4de7-993f-63a007958fad  in00000424929   
106  c9575ab0-b18a-453c-b4a8-aa93f802067f  in00000424930   
107  4bf86e10-a2b0-4383-99db-31245284b02c  in00000424963   
108  cebc135a-f549-425a-ae70-51f2eb117102  in00000425029   
109  d6d20349-05bc-45ed-baff-9b3b76d4a03f  in00000425062   

                                                 title  \
0    12 things to know about political parties / Vi...   
1    Writing the U.S. Constitution : framework of a...   
2       Bicycling the Colorado Rockies / Vici De Haan.   
3    Tōkyō Kokurit